In [1]:
# ============================================================
# 1) IMPORTS AND CPU SETTINGS
# ============================================================

import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

from pathlib import Path
import gc
import random
import time

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

# Restrict PyTorch CPU parallelism
try:
    torch.set_num_threads(1)
except Exception:
    pass

try:
    torch.set_num_interop_threads(1)
except Exception:
    pass

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# CPU ONLY for maximum stability
device = torch.device("cpu")

print("Device          :", device)
print("CPU threads     :", torch.get_num_threads())

Device          : cpu
CPU threads     : 1


In [2]:
TRAIN_SAMPLES = 60000
TEST_SAMPLES  = 10000

BATCH_SIZE = 100
# 60000 / 100 = 600 -> in per epoch we have 600 batch
EPOCHS = 2
LEARNING_RATE =0.05  # eta in manual SGD: W_new = W_old - eta * dL/dW

# SNN architecture
N_IN  = 28 * 28
N_H1  = 48
N_H2  = 24
N_OUT = 10

# SNN temporal parameters
T = 8
alpha = 0.80
beta = 0.95
theta = 1.0

# Surrogate-gradient slope
SG_SLOPE = 8.0

print("Architecture  :", f"{N_IN} -> {N_H1} -> {N_H2} -> {N_OUT}")

Architecture  : 784 -> 48 -> 24 -> 10


In [3]:
# ============================================================
# 3) LOAD ONLY A SMALL PART OF LOCAL MNIST CSV
# ============================================================
# Expected row format:
# label,pixel1,pixel2,...,pixel784
#
# The function stops immediately after reading the requested
# number of rows. It never loads the full 60,000-row CSV.

TRAIN_CSV = Path("main_train.csv")
TEST_CSV  = Path("main_test.csv")

for file_path in (TRAIN_CSV, TEST_CSV):
    if not file_path.exists():
        raise FileNotFoundError(
            f"{file_path.name} was not found.\n"
            f"Current folder: {Path.cwd()}\n"
            "Put main_train.csv and main_test.csv next to this notebook."
        )

def load_first_mnist_rows(path, n_rows):
    data = np.empty((n_rows, 785), dtype=np.uint8)
    filled = 0

    with open(path, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            # Temporary row only. int16 safely holds 0..255.
            row = np.fromstring(line.strip(), sep=",", dtype=np.int16)

            # Gracefully skip an unexpected header line
            if row.size != 785:
                if line_number == 1:
                    continue
                raise ValueError(
                    f"Invalid row {line_number} in {path.name}: "
                    f"expected 785 values, got {row.size}"
                )

            if row.min() < 0 or row.max() > 255:
                raise ValueError(
                    f"Values outside 0..255 found in row {line_number}"
                )

            data[filled] = row.astype(np.uint8, copy=False)
            filled += 1

            if filled == n_rows:
                break

    if filled < n_rows:
        data = data[:filled].copy()

    return data

train_np = load_first_mnist_rows(TRAIN_CSV, TRAIN_SAMPLES)
test_np  = load_first_mnist_rows(TEST_CSV, TEST_SAMPLES)

# Keep images as uint8 in RAM.
# They are converted to float only one small batch at a time.
train_x = torch.from_numpy(train_np[:, 1:].copy()).view(-1, 1, 28, 28)
train_y = torch.from_numpy(train_np[:, 0].astype(np.int64, copy=True))

test_x = torch.from_numpy(test_np[:, 1:].copy()).view(-1, 1, 28, 28)
test_y = torch.from_numpy(test_np[:, 0].astype(np.int64, copy=True))

# Free NumPy buffers immediately
del train_np, test_np
gc.collect()

train_loader = DataLoader(
    TensorDataset(train_x, train_y),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
    drop_last=False,
)

test_loader = DataLoader(
    TensorDataset(test_x, test_y),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    drop_last=False,
)

print("Loaded train images:", tuple(train_x.shape), train_x.dtype)
print("Loaded test images :", tuple(test_x.shape), test_x.dtype)
print("Train memory       :", f"{train_x.nelement() * train_x.element_size() / 1024**2:.2f} MB")
print("Test memory        :", f"{test_x.nelement() * test_x.element_size() / 1024**2:.2f} MB")
print("DataLoaders ready.")

Loaded train images: (60000, 1, 28, 28) torch.uint8
Loaded test images : (10000, 1, 28, 28) torch.uint8
Train memory       : 44.86 MB
Test memory        : 7.48 MB
DataLoaders ready.


In [4]:
# ============================================================
# 4) SURROGATE SPIKE + WEIGHTS
# ============================================================
#
# Forward:
#     hard spike = Theta(U - theta) -> feedforward
#
# Backward:
#     gradient comes from sigmoid(slope * x) -> backwardpass use surrogate
#
# The detach trick preserves exact 0/1 spikes in the forward pass.

def surrogate_spike(x, slope=SG_SLOPE):
    hard = (x >= 0).to(x.dtype)
    soft = torch.sigmoid(slope * x)
    return soft + (hard - soft).detach()

def make_weight(n_in, n_out, gain=0.7):
    w = torch.empty(n_in, n_out, device=device)
    torch.nn.init.xavier_uniform_(w, gain=gain)
    w.requires_grad_(True)
    return w

W1   = make_weight(N_IN, N_H1, 0.7)
W2   = make_weight(N_H1, N_H2, 0.7)
Wout = make_weight(N_H2, N_OUT, 0.7)

print("W1   :", tuple(W1.shape))
print("W2   :", tuple(W2.shape))
print("Wout :", tuple(Wout.shape))

W1   : (784, 48)
W2   : (48, 24)
Wout : (24, 10)


In [5]:
'''def Forward(x):
    return torch.where(
        x >= 0,
        torch.ones_like(x),
        torch.zeros_like(x)
    )


def backward(x):
    S_G = 1/1-e^x
    return S_G'''




'def Forward(x):\n    return torch.where(\n        x >= 0,\n        torch.ones_like(x),\n        torch.zeros_like(x)\n    )\n\n\ndef backward(x):\n    S_G = 1/1-e^x\n    return S_G'

In [6]:
# ============================================================
# 5) SNN FORWARD PASS
# ============================================================
#
# Layer 1:
# I1[n+1] = alpha*I1[n] + S_in[n] @ W1 
# U1[n+1] = beta*U1[n] + I1[n+1]
#
# Layer 2:
# I2[n+1] = alpha*I2[n] + S1[n] @ W2
# U2[n+1] = beta*U2[n] + I2[n+1]
#
# Backpropagation through these T time steps is BPTT.

def snn_forward(images):
    # images arrive as uint8 [0..255].
    # Convert only this small batch to float [0..1].
    x = images.to(device=device, dtype=torch.float32)
    x = x.flatten(1).div(255.0)

    batch = x.shape[0]

    I1 = torch.zeros(batch, N_H1, device=device)
    U1 = torch.zeros(batch, N_H1, device=device)

    I2 = torch.zeros(batch, N_H2, device=device)
    U2 = torch.zeros(batch, N_H2, device=device)

    logits_sum = torch.zeros(batch, N_OUT, device=device)

    for _ in range(T):
        # Rate/Poisson-style Bernoulli input spikes
        S_in = torch.bernoulli(x)

        # Hidden layer 1
        I1 = alpha * I1 + S_in @ W1#torch.matmul(S_in*W1),torch.dot
        '''
        outputs = []
        for neuron in range(48):
            y = torch.dot(
                S_in[0],
                W1.T[neuron]
            )
            outputs.append(y)
        '''
        U1 = beta * U1 + I1
        S1 = surrogate_spike(U1 - theta)
        U1 = U1 - theta * S1.detach()

        # Hidden layer 2
        I2 = alpha * I2 + S1 @ W2
        U2 = beta * U2 + I2
        S2 = surrogate_spike(U2 - theta)
        U2 = U2 - theta * S2.detach()

        # Non-spiking linear readout
        logits_sum = logits_sum + S2 @ Wout

    return logits_sum / T

In [7]:
# ============================================================
# 6) TRAINING AND TESTING - MANUAL SGD
# ============================================================
#
# We do NOT use torch.optim here.
#
# Backward:
#     loss.backward()
# computes gradients by BPTT + surrogate gradient.
#
# Manual SGD update:
#     W_new = W_old - eta * dL/dW
#
# where:
#     eta = LEARNING_RATE
#
# Each batch graph is destroyed before the next batch.

weights = [W1, W2, Wout]


@torch.no_grad()
def evaluate(loader):
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in loader:
        labels = labels.to(device)

        # Forward pass
        logits = snn_forward(images)

        # Cross-entropy loss
        loss = F.cross_entropy(logits, labels)

        batch_n = labels.numel()

        total_loss += loss.item() * batch_n
        total_correct += (
            logits.argmax(dim=1) == labels
        ).sum().item()
        total_samples += batch_n

        del logits, loss

    return (
        total_loss / total_samples,
        100.0 * total_correct / total_samples,
    )


history = []

print("Training started...\n")


for epoch in range(1, EPOCHS + 1):

    start_time = time.time()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0


    for images, labels in train_loader:

        labels = labels.to(device)


        for W in weights:
            W.grad = None


        # ----------------------------------------------------
        # 2) FORWARD PASS
        # ----------------------------------------------------

        logits = snn_forward(images)


        # ----------------------------------------------------
        # 3) LOSS
        # ----------------------------------------------------

        loss = F.cross_entropy(
            logits,
            labels
        )


        # ----------------------------------------------------
        # 4) BACKWARD PASS
        # ----------------------------------------------------
        #
        # snn_forward() contains T time steps, so autograd
        # propagates gradients through time (BPTT).
        #
        # At the spike function, surrogate_spike() provides
        # the sigmoid surrogate gradient.

        loss.backward()


        # ----------------------------------------------------
        # 5) MANUAL SGD UPDATE
        # ----------------------------------------------------
        #
        #     W_new = W_old - eta * dL/dW
        #
        # After loss.backward(), dL/dW is stored in W.grad.
        # The update itself must not be tracked by autograd.

        with torch.no_grad():

            W1 -= LEARNING_RATE * W1.grad
            W2 -= LEARNING_RATE * W2.grad
            Wout -= LEARNING_RATE * Wout.grad


        # ----------------------------------------------------
        # 6) TRAINING STATISTICS
        # ----------------------------------------------------

        batch_n = labels.numel()

        total_loss += loss.item() * batch_n

        total_correct += (
            logits.argmax(dim=1) == labels
        ).sum().item()

        total_samples += batch_n


        del logits, loss


    # --------------------------------------------------------
    # END OF EPOCH
    # --------------------------------------------------------

    train_loss = total_loss / total_samples

    train_acc = (
        100.0 * total_correct / total_samples
    )


    test_loss, test_acc = evaluate(test_loader)


    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "test_loss": test_loss,
        "test_acc": test_acc,
    })


    elapsed = time.time() - start_time


    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"train loss={train_loss:.4f} | "
        f"train acc={train_acc:.2f}% | "
        f"test loss={test_loss:.4f} | "
        f"test acc={test_acc:.2f}% | "
        f"time={elapsed:.1f}s"
    )


    gc.collect()


print("\nTraining finished successfully.")


Training started...

Epoch 1/2 | train loss=0.8232 | train acc=84.46% | test loss=0.3961 | test acc=91.87% | time=19.9s
Epoch 2/2 | train loss=0.3400 | train acc=92.05% | test loss=0.2836 | test acc=92.93% | time=18.2s

Training finished successfully.


In [8]:
# ============================================================
# 7) FINAL TEST + SIMPLE RESULT ANALYSIS
# ============================================================

images, labels = next(iter(test_loader))

with torch.no_grad():
    logits = snn_forward(images)
    predictions = logits.argmax(dim=1)


# ------------------------------------------------------------
# A) show samples result
# ------------------------------------------------------------

n_show = 100

print("=" * 60)
print("FINAL TEST - SAMPLE PREDICTIONS")
print("=" * 60)

correct_count = 0
wrong_count = 0

for i in range(n_show):

    true_label = int(labels[i].item())
    pred_label = int(predictions[i].item())

    if true_label == pred_label:
        status = "CORRECT"
        correct_count += 1
    else:
        status = "WRONG"
        wrong_count += 1

    print(
        f"Sample {i+1:02d} | "
        f"True = {true_label} | "
        f"Pred = {pred_label} | "
        f"{status}"
    )


# ------------------------------------------------------------
# B) result of final test samples
# ------------------------------------------------------------

sample_accuracy = 100.0 * correct_count / n_show

print("\n" + "-" * 60)
print("SAMPLE SUMMARY")
print("-" * 60)

print(f"Correct predictions : {correct_count}")
print(f"Wrong predictions   : {wrong_count}")
print(f"Sample accuracy     : {sample_accuracy:.2f}%")


# ------------------------------------------------------------
# C) final test set
# ------------------------------------------------------------

final_test_loss, final_test_acc = evaluate(test_loader)

print("\n" + "=" * 60)
print("FINAL TEST SET RESULT")
print("=" * 60)

print(f"Final test loss     : {final_test_loss:.4f}")
print(f"Final test accuracy : {final_test_acc:.2f}%")

FINAL TEST - SAMPLE PREDICTIONS
Sample 01 | True = 7 | Pred = 7 | CORRECT
Sample 02 | True = 2 | Pred = 2 | CORRECT
Sample 03 | True = 1 | Pred = 1 | CORRECT
Sample 04 | True = 0 | Pred = 0 | CORRECT
Sample 05 | True = 4 | Pred = 4 | CORRECT
Sample 06 | True = 1 | Pred = 1 | CORRECT
Sample 07 | True = 4 | Pred = 4 | CORRECT
Sample 08 | True = 9 | Pred = 9 | CORRECT
Sample 09 | True = 5 | Pred = 6 | WRONG
Sample 10 | True = 9 | Pred = 9 | CORRECT
Sample 11 | True = 0 | Pred = 0 | CORRECT
Sample 12 | True = 6 | Pred = 2 | WRONG
Sample 13 | True = 9 | Pred = 9 | CORRECT
Sample 14 | True = 0 | Pred = 0 | CORRECT
Sample 15 | True = 1 | Pred = 1 | CORRECT
Sample 16 | True = 5 | Pred = 5 | CORRECT
Sample 17 | True = 9 | Pred = 9 | CORRECT
Sample 18 | True = 7 | Pred = 7 | CORRECT
Sample 19 | True = 3 | Pred = 2 | WRONG
Sample 20 | True = 4 | Pred = 4 | CORRECT
Sample 21 | True = 9 | Pred = 9 | CORRECT
Sample 22 | True = 6 | Pred = 6 | CORRECT
Sample 23 | True = 6 | Pred = 6 | CORRECT
Sample 2